# RVV vectorization

triton-riscv targets the RISC-V vector extension by default: its `llc` feature
string is `+m,+a,+f,+d,+c,+v,+zfh,+zvfh,+zba,+zbb`, and the `+v` in there is
what turns a `tl.arange` loop into `vle32.v` / `vfadd.vv` / `vse32.v` under a
`vsetivli`/`vsetvli` vector-length setting.

This notebook compiles one kernel twice -- once for RVV and once for the plain
scalar baseline -- and reads the RISC-V assembly of both. Compiling is enough,
nothing has to run, which is why this works on an x86 host just as well as on a
RISC-V one: `target_triple` is what decides which machine the backend emits
code for.


## Activate the backend

triton-riscv installs itself as a Triton plugin under the name
`triton_shared`. On a CPU-only host Triton picks it on its own, but saying so
explicitly keeps these kernels on the CPU backend on machines that also have a
GPU, where the CUDA backend would otherwise win:


In [1]:
import triton
from triton.backends.triton_shared.driver import CPUDriver

triton.runtime.driver.set_active(CPUDriver())

In [2]:
import re
import subprocess
import tempfile
from pathlib import Path

import torch

import triton.language as tl

RVV_FEATURES = "+m,+a,+f,+d,+c,+v,+zfh,+zvfh,+zba,+zbb"
SCALAR_FEATURES = "+m,+a,+f,+d"


@triton.jit
def add_kernel(x_ptr, y_ptr, out_ptr, n_elements, BLOCK_SIZE: tl.constexpr):
    pid = tl.program_id(axis=0)
    offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n_elements
    x = tl.load(x_ptr + offsets, mask=mask)
    y = tl.load(y_ptr + offsets, mask=mask)
    tl.store(out_ptr + offsets, x + y, mask=mask)

## Compile for RISC-V without running it

`warmup` compiles a kernel and hands back the compiled kernel; `target_triple`
and `target_features` are ordinary backend options, so the same call produces
RISC-V code on any host.


In [3]:
def compile_for(features: str):
    n = 4096
    x = torch.randn(n)
    y = torch.randn(n)
    out = torch.empty_like(x)
    return add_kernel.warmup(
        x,
        y,
        out,
        n,
        BLOCK_SIZE=1024,
        grid=(triton.cdiv(n, 1024),),
        target_triple="riscv64-unknown-linux-gnu",
        target_features=features,
    )


def disassemble(kernel) -> str:
    with tempfile.TemporaryDirectory() as directory:
        path = Path(directory) / "kernel.o"
        path.write_bytes(kernel.asm["obj"])
        return subprocess.check_output(["llvm-objdump", "-d", str(path)], text=True)


rvv = compile_for(RVV_FEATURES)
scalar = compile_for(SCALAR_FEATURES)

for label, kernel in (("rvv", rvv), ("scalar", scalar)):
    asm = disassemble(kernel)
    counts = {
        op: len(re.findall(rf"\b{op}\b", asm))
        for op in ("vsetivli", "vle32.v", "vse32.v", "vfadd.vv")
    }
    print(f"{label:7} {counts}")

rvv     {'vsetivli': 1, 'vle32.v': 2, 'vse32.v': 1, 'vfadd.vv': 1}
scalar  {'vsetivli': 0, 'vle32.v': 0, 'vse32.v': 0, 'vfadd.vv': 0}


The same instructions, this time as they appear in the disassembly:


In [4]:
for line in disassemble(rvv).splitlines():
    if re.search(r"\b(vset|vle|vse|vfadd)", line):
        print(line.replace("\t", " ").strip())

42: cd287057      vsetivli zero, 0x10, e32, m4, ta, ma
46: 020ee407      vle32.v v8, (t4)
4e: 020ee607      vle32.v v12, (t4)
52: 02861457      vfadd.vv v8, v8, v12
60: 020ee427      vse32.v v8, (t4)


With `+v`, the two loads, the add and the store all go through the vector unit.
The block size is a compile-time constant here, so the code sets the vector
length with the immediate form `vsetivli`; a trip count that is only known at
run time produces the register form `vsetvli` instead. Without `+v` the
compiler falls back to scalar instructions and none of these mnemonics appear.


In [5]:
for op in ("vsetivli", "vle32.v", "vse32.v", "vfadd.vv"):
    assert len(re.findall(rf"\b{op}\b", disassemble(rvv))), (
        f"{op} missing from the RVV build"
    )
    assert not re.search(rf"\b{op}\b", disassemble(scalar)), f"{op} in the scalar build"

## The IR it came from

`warmup` also keeps every stage it went through, so when the assembly is not
what you expected you can walk back up: `ttir` (Triton IR), `ttsharedir`
(triton-shared's structured IR), `llir` (the LLVM IR handed to `llc`) and `obj`.


In [6]:
print(sorted(rvv.asm.keys()))

mlir = rvv.asm["ttsharedir"].splitlines()
start = next(i for i, line in enumerate(mlir) if line.startswith("module {"))
print("\n".join(line[:150] for line in mlir[start : start + 10]))

['llir', 'obj', 'source', 'ttir', 'ttsharedir']
module {
  func.func @add_kernel(%arg0: memref<*xf32> {tt.divisibility = 16 : i32} loc("x_ptr"(#loc)), %arg1: memref<*xf32> {tt.divisibility = 16 : i32} loc("y
    %c1024_i32 = arith.constant 1024 : i32 loc(#loc11)
    %c1024 = arith.constant 1024 : index loc(#loc12)
    %c0 = arith.constant 0 : index loc(#loc3)
    %c1 = arith.constant 1 : index loc(#loc3)
    %c16 = arith.constant 16 : index loc(#loc3)
    %0 = arith.muli %arg7, %c1024_i32 : i32 loc(#loc11)
    %1 = arith.index_cast %0 : i32 to index loc(#loc4)
    %2 = arith.addi %1, %c1024 : index loc(#loc12)
